# EAGF Notebook 4: Pareto-Front MOO Visualisation

This notebook demonstrates and visualises the multi-objective optimisation (MOO)
used in EAGF (Paper Section 3.7):
- Sweep of the 5×5 (lambda_RP × lambda_C) Lagrangian grid
- Pareto-front identification (non-dominated sorting)
- Privacy–Fairness trade-off surface
- Selection of the best-TI model from the Pareto front

**Note:** Full 25-run grid is run here with fewer epochs for speed. Use
`python run_eagf.py --epochs 50` for paper-quality results.

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yaml

print('Imports ready.')

## 1. Run 5×5 Pareto Grid Search

In [ ]:
from src.training.pareto_trainer import run_pareto_search
from src.utils.data_loader import generate_demo_biometric

with open(os.path.join(PROJECT_ROOT, 'configs', 'biometric_default.yaml')) as f:
    config = yaml.safe_load(f)
config['training']['epochs'] = 20  # fast for notebook

dataset = generate_demo_biometric(n_samples=800, seed=42)

print('Running 5×5 Pareto grid (25 training runs)...')
pareto_result = run_pareto_search(
    config=config,
    lambda_rp_range=(1e-3, 1.0),
    lambda_c_range=(1e-3, 1.0),
    n_steps=5,
    seed=42,
    device='cpu',
    output_dir='/tmp/eagf_nb4/pareto',
    dataset=dataset,
)

all_r = pareto_result['all_results']
front = pareto_result['pareto_front']
best  = pareto_result['best']

print(f'\nTotal runs        : {len(all_r)}')
print(f'Pareto-optimal    : {len(front)}')
print(f'Best TI           : {best["trust_index"]:.3f} '
      f'(lambda_RP={best["lambda_rp"]:.4f}, lambda_C={best["lambda_c"]:.4f})')

## 2. Privacy–Fairness Trade-Off Surface

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Privacy vs Recall Parity scatter
ax = axes[0]
dom_pts  = [r for r in all_r if r not in front]
par_pts  = front

if dom_pts:
    ax.scatter([r['privacy'] for r in dom_pts],
               [r['recall_parity'] for r in dom_pts],
               c='lightgrey', s=50, label=f'Dominated ({len(dom_pts)})', zorder=2)

ax.scatter([r['privacy'] for r in par_pts],
           [r['recall_parity'] for r in par_pts],
           c='#3CB371', s=80, label=f'Pareto-optimal ({len(par_pts)})', zorder=3)

ax.scatter(best['privacy'], best['recall_parity'],
           c='red', s=200, marker='*', label=f'Selected (max TI={best["trust_index"]:.3f})', zorder=4)

ax.set_xlabel('Privacy (P)', fontsize=11)
ax.set_ylabel('Recall Parity (RP)', fontsize=11)
ax.set_title('Pareto Front: Privacy vs. Fairness Trade-off', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.2)
ax.spines[['top','right']].set_visible(False)

# Right: TI colourmap over the lambda grid
ax2 = axes[1]
try:
    lrp_vals = sorted(set(round(r['lambda_rp'], 6) for r in all_r))
    lc_vals  = sorted(set(round(r['lambda_c'],  6) for r in all_r))
    TI_grid  = np.zeros((len(lrp_vals), len(lc_vals)))
    for r in all_r:
        i = lrp_vals.index(round(r['lambda_rp'],6))
        j = lc_vals.index(round(r['lambda_c'],6))
        TI_grid[i, j] = r['trust_index']

    im = ax2.imshow(TI_grid, aspect='auto', origin='lower', cmap='RdYlGn', vmin=0.4, vmax=1.0)
    plt.colorbar(im, ax=ax2, label='Trust Index (TI)')
    ax2.set_xlabel('lambda_C (log scale)', fontsize=11)
    ax2.set_ylabel('lambda_RP (log scale)', fontsize=11)
    ax2.set_xticks(range(len(lc_vals)))
    ax2.set_xticklabels([f'{v:.3f}' for v in lc_vals], rotation=45, fontsize=8)
    ax2.set_yticks(range(len(lrp_vals)))
    ax2.set_yticklabels([f'{v:.3f}' for v in lrp_vals], fontsize=8)
    ax2.set_title('Trust Index Heatmap over Lambda Grid', fontsize=11)

    # Mark best solution
    bi = lrp_vals.index(round(best['lambda_rp'],6))
    bj = lc_vals.index(round(best['lambda_c'],6))
    ax2.plot(bj, bi, 'r*', markersize=15, label='Best TI')
    ax2.legend(fontsize=9)
except Exception as e:
    ax2.text(0.5, 0.5, f'Heatmap unavailable:\n{e}', ha='center', va='center',
             transform=ax2.transAxes)

plt.tight_layout()
out = os.path.join(PROJECT_ROOT, 'figures', 'notebook4_pareto.png')
plt.savefig(out, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out}')

## 3. Cross-Pillar Trade-Off Analysis

In [ ]:
# Show how each objective changes as lambda_RP increases (lambda_C fixed at median)
import pandas as pd

lc_med   = sorted(set(round(r['lambda_c'],6) for r in all_r))[2]  # middle column
lrp_seq  = [r for r in sorted(all_r, key=lambda x: x['lambda_rp'])
             if abs(round(r['lambda_c'],6) - lc_med) < 1e-6]

rows = []
for r in lrp_seq:
    rows.append({
        'lambda_RP':      f"{r['lambda_rp']:.4f}",
        'Accuracy':       round(r.get('accuracy', 0), 3),
        'Recall Parity':  round(r.get('recall_parity', 0), 3),
        'Clarity (C)':    round(r.get('clarity', 0), 3),
        'Privacy (P)':    round(r.get('privacy', 0), 3),
        'Trust Index':    round(r.get('trust_index', 0), 3),
    })

df = pd.DataFrame(rows).set_index('lambda_RP')
print(f'Effect of lambda_RP on all metrics (lambda_C = {lc_med:.4f} fixed):')
print('=' * 65)
print(df.to_string())
print()
print('Observation: Higher lambda_RP → higher RP but slight accuracy cost.')
print('EAGF Pareto selection balances all four pillars simultaneously.')